In [ ]:
import json
import subprocess
import re
from pathlib import Path
import sys
import tarfile # Importa a biblioteca para manipular arquivos .tar.gz
import os # Importa a biblioteca os para manipulação de caminhos

# Caminho raiz do projeto (onde está o pyproject.toml)
PROJECT_DIR = Path("./codebench-analytics-full").resolve()

# O diretório de extraídos está um nível acima do local atual.
EXTRAIDOS_DIR = Path("../Extraidos").resolve()


def extract_tar_gz_files(directory):
    """
    Procura por arquivos .tar.gz em um diretório e os extrai.
    Remove o arquivo .tar.gz após a extração bem-sucedida.
    """
    for item in os.listdir(directory):
        if item.endswith(".tar.gz"):
            tar_path = os.path.join(directory, item)
            print(f"📦 Encontrado arquivo compactado: {tar_path}. Extraindo...")
            try:
                with tarfile.open(tar_path, "r:gz") as tar:
                    # Extrai os arquivos para o mesmo diretório
                    tar.extractall(path=directory)
                print(f"✅ Extração de {item} concluída.")
                # Remove o arquivo .tar.gz após a extração
                os.remove(tar_path)
                print(f"🗑️ Arquivo {item} removido.")
            except Exception as e:
                print(f"❌ Erro ao extrair {tar_path}: {e}")


def extract_year_semester_from_folder(folder_name):
    """
    Extrai ano e semestre do nome da pasta.
    Exemplo: cb_dataset_2018_1_v1.81 -> (2018, 1)
    """
    match = re.search(r'cb_dataset_(\d{4})_(\d+)_v', folder_name)
    if match:
        year = match.group(1)
        semester = match.group(2)
        return year, semester
    return None, None

def discover_datasets():
    """
    Descobre, descompacta e prepara os caminhos dos datasets.
    """
    print(f"Procurando datasets em: {EXTRAIDOS_DIR}")
    
    if not EXTRAIDOS_DIR.exists():
        print(f"❌ Diretório Extraidos não encontrado: {EXTRAIDOS_DIR}")
        sys.exit(1)
    
    dataset_paths = []
    
    # Procurar por pastas que seguem o padrão cb_dataset_*
    for item in EXTRAIDOS_DIR.iterdir():
        if item.is_dir() and item.name.startswith("cb_dataset_"):
            year, semester = extract_year_semester_from_folder(item.name)
            
            if year and semester:
                # Construir o caminho para a pasta interna
                dataset_path = item / f"{year}-{semester}"
                
                if dataset_path.exists():
                    # --- NOVA ETAPA: VERIFICAR E EXTRAIR ARQUIVOS .TAR.GZ ---
                    extract_tar_gz_files(dataset_path)
                    
                    dataset_paths.append(str(dataset_path.resolve()))
                    print(f"✅ Dataset preparado: {item.name} -> {dataset_path}")
                else:
                    print(f"⚠️  Pasta interna não encontrada: {dataset_path}")
            else:
                print(f"⚠️  Não foi possível extrair ano/semestre de: {item.name}")
    
    if not dataset_paths:
        print("❌ Nenhum dataset válido encontrado!")
        sys.exit(1)
    
    # Ordenar os caminhos para processamento consistente
    dataset_paths.sort()
    
    print(f"\n📊 Total de datasets preparados: {len(dataset_paths)}")
    return dataset_paths

dataset_paths = discover_datasets()

# Controle persistente: não apagar outputs anteriores e permitir retomada segura.
CONTROL_PATH = PROJECT_DIR.parent / "output" / "controle_processamento.json"

def carregar_controle():
    if CONTROL_PATH.exists():
        try:
            return json.loads(CONTROL_PATH.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            pass
    return {"versao": 1, "datasets": {}, "etapas": {}}

def salvar_controle(controle):
    CONTROL_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporario = CONTROL_PATH.with_suffix(".tmp")
    temporario.write_text(json.dumps(controle, ensure_ascii=False, indent=2), encoding="utf-8")
    temporario.replace(CONTROL_PATH)

def atualizar_controle_etapa(nome, status, datasets=None, erro=None):
    controle = carregar_controle()
    registro = controle.setdefault("etapas", {}).setdefault(nome, {})
    registro.update({"status": status, "atualizado_em": __import__("datetime").datetime.now().isoformat(timespec="seconds")})
    if datasets is not None:
        registro["datasets"] = list(datasets)
    if erro:
        registro["erro"] = str(erro)
    elif status == "concluido":
        registro.pop("erro", None)
    salvar_controle(controle)

def etapa_concluida(nome):
    return carregar_controle().get("etapas", {}).get(nome, {}).get("status") == "concluido"





In [ ]:
import subprocess
import json

def executar_com_progresso(command, cwd):
    """Executa o comando e retransmite stdout/stderr imediatamente."""
    processo = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    try:
        for linha in processo.stdout:
            print(linha, end="", flush=True)
    except KeyboardInterrupt:
        print("\n⚠️ Interrupção solicitada: encerrando o processo filho com segurança...")
        processo.terminate()
        try:
            processo.wait(timeout=10)
        except subprocess.TimeoutExpired:
            processo.kill()
            processo.wait()
        raise
    finally:
        if processo.stdout is not None:
            processo.stdout.close()
    return processo.wait()

import sys
import os
from pathlib import Path

def codebench_analytics_bin() -> str:
    """
    Caminho para o executavel 'codebench_analytics' instalado no MESMO venv
    que esta rodando este notebook (instalado via 'pip install -e' pelo
    setup_env.py da raiz do projeto -- nao depende de Poetry nem de 'make').
    """
    bin_dir = Path(sys.executable).parent
    name = "codebench_analytics.exe" if os.name == "nt" else "codebench_analytics"
    exe = bin_dir / name
    if not exe.exists():
        print(f"❌ Nao encontrei '{name}' em {bin_dir}.")
        print("   Rode 'python setup_env.py install' na raiz do projeto para instalar")
        print("   o sub-projeto Etapa_4/codebench-analytics-full no venv atual.")
        sys.exit(1)
    return str(exe)


def _logging_tree(workdir):
    import shutil
    logging_src = PROJECT_DIR / "codebench_analytics" / "logging"
    logging_dst = workdir / "codebench_analytics" / "logging"
    shutil.copytree(logging_src, logging_dst)


def _csv_output(workdir, name):
    return workdir / "output" / "data" / name


def _run_isolated_dataset(kind, dataset_path):
    """Extrai somente eventos brutos; o collector roda uma única vez no consolidado."""
    import tempfile
    import os
    import shutil
    workdir = Path(tempfile.mkdtemp(prefix=f"codebench_{kind}_"))
    try:
        _logging_tree(workdir)
        cwd = os.getcwd()
        try:
            os.chdir(workdir)
            from codebench_analytics.utils.assessments_filter import AssessmentType
            from codebench_analytics.model.codebench_types import Resource
            if kind == "execution":
                from codebench_analytics.extractor.execution_extractor import ExecutionExtractor
                raw = Path(ExecutionExtractor(dataset_path, resource=Resource.EXECUTIONS).extract_from([AssessmentType.EXAM]))
            else:
                from codebench_analytics.extractor.action_extractor import ActionsExtractor
                raw = Path(ActionsExtractor(dataset_path, resource=Resource.CODEMIRRORS).extract_from([AssessmentType.EXAM]))
        finally:
            os.chdir(cwd)
        if not raw.exists():
            raise RuntimeError(f"{kind} não gerou CSV bruto para {dataset_path}")
        return dataset_path, workdir / raw
    except Exception:
        shutil.rmtree(workdir, ignore_errors=True)
        raise


def _consolidar_csvs(csv_paths, destino):
    import csv
    destino.parent.mkdir(parents=True, exist_ok=True)
    cabecalho = None
    temporario = destino.with_suffix(destino.suffix + ".tmp")
    with temporario.open("w", encoding="utf-8", newline="") as f_out:
        writer = None
        for csv_path in csv_paths:
            with Path(csv_path).open("r", encoding="utf-8", newline="") as f_in:
                reader = csv.DictReader(f_in)
                if cabecalho is None:
                    cabecalho = reader.fieldnames or []
                    writer = csv.DictWriter(f_out, fieldnames=cabecalho)
                    writer.writeheader()
                elif reader.fieldnames != cabecalho:
                    raise RuntimeError(f"Cabeçalhos incompatíveis em {csv_path}")
                for row in reader:
                    writer.writerow(row)
        if writer is None:
            temporario.write_text("", encoding="utf-8")
    temporario.replace(destino)


def _staging(dataset, nome):
    return PROJECT_DIR.parent / "output" / f"{nome}_staging" / (Path(dataset).name + ".csv")


def _consolidar_e_coletar(kind, staging_paths, combined):
    import os
    _consolidar_csvs(staging_paths, combined)
    cwd = os.getcwd()
    try:
        os.chdir(PROJECT_DIR)
        if kind == "execution":
            from codebench_analytics.collector.executions import ExecutionCollector
            ExecutionCollector(str(combined)).collect()
        else:
            from codebench_analytics.collector.actions import ActionCollector
            ActionCollector(str(combined)).collect()
    finally:
        os.chdir(cwd)


def _run_staged_kind(kind, paths):
    nome = "execution" if kind == "execution" else "action"
    if etapa_concluida(nome):
        print(f"✅ {nome} já concluída segundo controle_processamento.json; etapa ignorada.")
        return
    print(f"Executando análise de {nome} por dataset, com retomada individual:")
    for p in paths:
        print(f" - {p}")
    controle = carregar_controle()
    registro = controle.setdefault("etapas", {}).setdefault(nome, {})
    concluidos = set(registro.get("datasets_concluidos", []))
    # Compatibilidade com o controle antigo, que guardava apenas 'datasets'.
    staging_dir = PROJECT_DIR.parent / "output" / f"{nome}_staging"
    staging_dir.mkdir(parents=True, exist_ok=True)
    # Nunca confiar apenas no controle antigo: só reutilizar um dataset quando
    # o parcial correspondente existir fisicamente e puder ser consolidado.
    concluidos = {
        p for p in concluidos
        if _staging(p, nome).is_file()
    }
    for p in paths:
        if _staging(p, nome).is_file():
            concluidos.add(p)
    try:
        for dataset_path in paths:
            if dataset_path in concluidos:
                print(f"⏭️ Dataset {dataset_path} já concluído; parcial reutilizado.")
                continue
            print(f"\n▶ Dataset {nome}: {dataset_path}")
            _, raw = _run_isolated_dataset(kind, dataset_path)
            import shutil
            destino = _staging(dataset_path, nome)
            shutil.copy2(raw, destino)
            concluidos.add(dataset_path)
            atualizar_controle_etapa(nome, "em_andamento", sorted(concluidos))
            # O campo explícito facilita inspeção humana e compatibilidade futura.
            controle = carregar_controle()
            controle.setdefault("etapas", {}).setdefault(nome, {})["datasets_concluidos"] = sorted(concluidos)
            salvar_controle(controle)
            print(f"✅ Dataset concluído: {dataset_path}; parcial salvo em {destino}")
        if all(p in concluidos for p in paths):
            staging_paths = [_staging(p, nome) for p in paths]
            missing = [str(p) for p in staging_paths if not p.exists()]
            if missing:
                raise RuntimeError(f"Parciais ausentes para consolidação: {missing}")
            combined = PROJECT_DIR / "output" / "data" / ("executions_data.csv" if kind == "execution" else "actions.csv")
            _consolidar_e_coletar(kind, staging_paths, combined)
            atualizar_controle_etapa(nome, "concluido", sorted(concluidos))
            controle = carregar_controle()
            controle.setdefault("etapas", {}).setdefault(nome, {})["datasets_concluidos"] = sorted(concluidos)
            salvar_controle(controle)
            print(f"✅ {nome.capitalize()} consolidada e registrada no controle.")
    except KeyboardInterrupt:
        atualizar_controle_etapa(nome, "interrompido", sorted(concluidos), "Interrompido pelo usuário")
        raise
    except Exception as exc:
        atualizar_controle_etapa(nome, "falhou", sorted(concluidos), exc)
        raise


def run_execution_command(*paths):
    _run_staged_kind("execution", paths)


def run_action_command(*paths):
    _run_staged_kind("action", paths)


def run_solution_command(solution_path):
    if etapa_concluida("solution"):
        print("✅ solution já concluída segundo controle_processamento.json; etapa ignorada.")
        return
    print(f"Extraindo métricas da solução: {solution_path}")
    atualizar_controle_etapa("solution", "em_andamento", [solution_path])
    import shutil
    import tempfile
    workdir = Path(tempfile.mkdtemp(prefix="codebench_solution_"))
    _logging_tree(workdir)
    command = [codebench_analytics_bin(), "solution", "-p", solution_path]
    try:
        returncode = executar_com_progresso(command, workdir)
        raw = workdir / "output" / "data" / "code_metrics_professor.csv"
        if returncode != 0 or not raw.exists():
            raise RuntimeError(f"solution falhou (código {returncode})")
        staging = PROJECT_DIR.parent / "output" / "solution_staging" / "code_metrics_professor.csv"
        staging.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(raw, staging)
        destino = PROJECT_DIR / "output" / "data" / "code_metrics_professor.csv"
        shutil.copy2(staging, destino)
        atualizar_controle_etapa("solution", "concluido", [solution_path])
        print(f"✅ Métricas da solução concluídas; parcial salvo em {staging}.")
    except KeyboardInterrupt:
        atualizar_controle_etapa("solution", "interrompido", [solution_path], "Interrompido pelo usuário")
        raise
    except Exception as exc:
        atualizar_controle_etapa("solution", "falhou", [solution_path], exc)
        raise
    finally:
        shutil.rmtree(workdir, ignore_errors=True)


def main():
    run_execution_command(*dataset_paths)
    run_action_command(*dataset_paths)
    solution_csv = PROJECT_DIR / "input" / "codigo_solucao.csv"
    if not solution_csv.exists():
        print(f"❌ Arquivo de solução não encontrado: {solution_csv}")
        sys.exit(1)
    run_solution_command(str(solution_csv))


if __name__ == "__main__":
    main()


# Explicação do Código

## O que o código faz:
1. **Carregamento de Arquivos CSV**:
   - Carrega dois arquivos CSV usando `pandas`:
     - `actions_data.csv`: contém informações sobre tempos e eventos relacionados a ações.
     - `executions.csv`: contém informações sobre interações e resultados relacionados às execuções.

2. **Renomeação de Colunas**:
   - Renomeia algumas colunas dos dois arquivos para facilitar a combinação e tornar os nomes mais descritivos:
     - No `actions_data.csv`:
       - `code_time` → `tempo_implementacao`
       - `num_events` → `num_eventos`
       - `num_deletes` → `num_eventos_del`
     - No `executions.csv`:
       - `num_submissions` → `num_submissoes`
       - `num_students_interactions` → `num_consultas`
       - `amount_of_change` → `qtd_alteracoes_codigo`

3. **União dos Dados**:
   - Combina os dois DataFrames com base na coluna `question`, utilizando um merge à esquerda (`how='left'`), o que preserva todas as linhas de `executions.csv`.

4. **Seleção de Colunas**:
   - Cria um novo DataFrame contendo apenas as colunas relevantes para a análise:
     - `question`, `tempo_implementacao`, `num_eventos`, `num_eventos_del`, `num_consultas`, `num_submissoes`, `num_tests`, `num_correct`, `num_errors`, `num_logic_errors`, `num_syntax_errors`, `qtd_alteracoes_codigo`.

5. **Exportação para CSV**:
   - Salva o DataFrame resultante em um novo arquivo chamado `resultado.csv`.

6. **Mensagem de Sucesso**:
   - Exibe uma mensagem no console indicando que o arquivo foi gerado com sucesso.

## Resultado:
- Um arquivo CSV chamado `resultado.csv`, contendo as colunas combinadas e selecionadas dos dois arquivos originais, está pronto para ser usado em análises ou relatórios.


In [ ]:
import pandas as pd
import os

# Cria a pasta de saída, caso ainda não exista
os.makedirs('output', exist_ok=True)

# Carregar os dois arquivos CSV
df1 = pd.read_csv(r'codebench-analytics-full/output/metrics/actions_data.csv')  # Primeiro arquivo com os tempos e eventos
df2 = pd.read_csv(r'codebench-analytics-full/output/metrics/executions.csv')  # Segundo arquivo com as interações e resultados

# Unir os DataFrames com base na coluna 'question'
merged_df = pd.merge(df2, df1, on='question', how='left')

# Calcular todas as métricas M1-M13 possíveis
merged_df['taxa_acerto']         = merged_df['num_correct'] / merged_df['num_students_interactions']        # M1
merged_df['num_submissoes']      = merged_df['num_submissions'] / merged_df['num_students_interactions']    # M2
merged_df['taxa_aceitacao']      = merged_df['num_correct'] / merged_df['num_submissions']                  # M3
merged_df['num_testes']          = merged_df['num_tests'] / merged_df['num_students_interactions']          # M4
merged_df['num_consultas']       = (merged_df['num_submissions'] + merged_df['num_tests']) / merged_df['num_students_interactions']  # M5
merged_df['num_erros_lgcs']      = merged_df['num_logic_errors'] / merged_df['num_students_interactions']   # M6
merged_df['num_errors_stx']      = merged_df['num_syntax_errors'] / merged_df['num_students_interactions']  # M7
merged_df['num_erros']           = merged_df['num_errors'] / merged_df['num_students_interactions']         # M8
merged_df['num_eventos']         = merged_df['num_events'] / merged_df['num_students_interactions']         # M9
merged_df['num_eventos_del']     = merged_df['num_deletes'] / merged_df['num_students_interactions']        # M10
merged_df['tempo_implementacao'] = merged_df['code_time'] / merged_df['num_correct']                        # M11
merged_df['qtd_alteracoes_codigo'] = merged_df['amount_of_change'] / merged_df['num_students_interactions'] # M12

# Selecionar colunas finais
final_df = merged_df[[
    'question',
    'taxa_acerto',           # M1
    'num_submissoes',        # M2
    'taxa_aceitacao',        # M3
    'num_testes',            # M4
    'num_consultas',         # M5
    'num_erros_lgcs',        # M6
    'num_errors_stx',        # M7
    'num_erros',             # M8
    'num_eventos',           # M9
    'num_eventos_del',       # M10
    'tempo_implementacao',   # M11
    'qtd_alteracoes_codigo', # M12
]]

final_df.to_csv('output/metricas_execucao_por_questao.csv', index=False)
print("Arquivo 'output/metricas_execucao_por_questao.csv' gerado com sucesso!")


## Explicação do Código

Este código tem como objetivo ler um arquivo CSV, manipular os dados e salvar um novo arquivo com os resultados. O processo é o seguinte:

1. **Leitura do arquivo CSV**: 
   O arquivo `'unified_solutions.csv'` é carregado em um DataFrame utilizando a biblioteca `pandas`.

2. **Criação de nova coluna**:
   Uma nova coluna chamada `'id'` é criada, copiando os valores da coluna `'assignment'`.

3. **Remoção de duplicatas**:
   A coluna `'id'` é filtrada para remover duplicatas, criando um novo DataFrame com valores únicos.

4. **Salvamento dos dados**:
   O novo DataFrame é salvo em um arquivo CSV chamado `'assessments.csv'`.

5. **Mensagem de sucesso**:
   Uma mensagem é exibida para informar que o arquivo de saída foi criado com sucesso.

### Código

```python
import pandas as pd

input_file = 'unified_solutions.csv'
output_file = 'assessments.csv'

df = pd.read_csv(input_file)
df['id'] = df['assignment']
df_output = df[['id']].drop_duplicates()

df_output.to_csv(output_file, index=False)

print(f"Arquivo '{output_file}' criado com sucesso!")


In [ ]:
import pandas as pd
import os

# Cria a pasta de saida, caso ainda nao exista
os.makedirs('output', exist_ok=True)
os.makedirs('input', exist_ok=True)

# Caminho do arquivo de entrada (externo, nao pertence a nenhuma etapa) e nome do arquivo de saida
input_file = os.path.join('..', 'CSVS_JO', 'unified_solutions.csv')  # gerado por Etapa_1/Etapa_1.ipynb
output_file = 'output/assessments_ids_unicos.csv'

# Carrega o CSV de entrada
df = pd.read_csv(input_file)

# Cria a coluna 'id' a partir da coluna 'assignment'
df['id'] = df['assignment']

# Remove duplicatas da coluna 'id'
df_output = df[['id']].drop_duplicates()

# Salva o novo DataFrame em um arquivo CSV
df_output.to_csv(output_file, index=False)

print(f"Arquivo '{output_file}' criado com sucesso!")


# Explicação do Código

## O que o código faz:

1. **Importação do Módulo CSV**:
   - Utiliza o módulo `csv` do Python para manipular arquivos CSV.

2. **Função para Leitura de CSV**:
   - A função `ler_csv`:
     - Recebe o nome de um arquivo CSV.
     - Lê o conteúdo do arquivo e retorna uma lista de dicionários, onde cada linha do CSV é representada como um dicionário.

3. **Leitura dos Arquivos de Entrada**:
   - `dados_question`: Contém informações sobre as questões (ex.: tempo de implementação, número de eventos, etc.), lido de `resultado.csv`.
   - `dados_respostas`: Contém informações adicionais sobre as questões (ex.: dificuldade, discriminação, etc.), lido de `questoes_ordenadas.csv`.

4. **Combinação dos Dados**:
   - Para cada questão em `dados_question`:
     - Procura a questão correspondente em `dados_respostas` (compara a coluna `id` com `question`).
     - Cria um novo registro com informações combinadas de ambos os arquivos.

5. **Estrutura do Novo Registro**:
   - As informações do novo registro incluem:
     - Dados da questão (`tempo_implementacao`, `num_eventos`, `num_tests`, etc.).
     - Dados adicionais das respostas (`dificuldade`, `discriminacao`, etc.).

6. **Escrita do Arquivo Resultante**:
   - Salva os registros combinados em um novo arquivo chamado `question_new_info.csv`:
     - Escreve um cabeçalho com os nomes das colunas.
     - Adiciona as linhas dos registros combinados.

7. **Mensagem de Sucesso**:
   - Exibe uma mensagem no console indicando que os dados foram combinados e salvos com sucesso.

## Resultado:
- Um arquivo CSV chamado `question_new_info.csv` é gerado, contendo informações combinadas de `resultado.csv` e `questoes_ordenadas.csv`.
- Esse arquivo pode ser usado para análises completas das questões e suas respectivas métricas.


In [ ]:
import csv
import os

os.makedirs('output', exist_ok=True)

def ler_csv(nome_arquivo):
    with open(nome_arquivo, mode='r', newline='') as arquivo_csv:
        leitor = csv.DictReader(arquivo_csv)
        return [linha for linha in leitor]

dados_question = ler_csv(r'output/metricas_execucao_por_questao.csv')
dados_respostas = ler_csv(r'../Etapa_3/output/questoes_ordenadas.csv')

dados_respostas = ler_csv(r'../Etapa_3/output/questoes_ordenadas.csv')
# Índice por ID: substitui a busca aninhada O(N*M) por buscas O(1),
# preservando a mesma regra de correspondência por questão.
respostas_por_id = {int(resposta['id']): resposta for resposta in dados_respostas}
resultados = []
for question in dados_question:
    resposta = respostas_por_id.get(int(question['question']))
    if resposta is None:
        continue
    novo_registro = {
        "question":              question['question'],
        "taxa_acerto":           question['taxa_acerto'],
        "num_submissoes":        question['num_submissoes'],
        "taxa_aceitacao":        question['taxa_aceitacao'],
        "num_testes":            question['num_testes'],
        "num_consultas":         question['num_consultas'],
        "num_erros_lgcs":        question['num_erros_lgcs'],
        "num_errors_stx":        question['num_errors_stx'],
        "num_erros":             question['num_erros'],
        "num_eventos":           question['num_eventos'],
        "num_eventos_del":       question['num_eventos_del'],
        "tempo_implementacao":   question['tempo_implementacao'],
        "qtd_alteracoes_codigo": question['qtd_alteracoes_codigo'],
        "dificuldade":           resposta['dificuldade'],
        "discriminacao":         resposta['discriminacao'],
        "respostas":             resposta['respostas'],
        "usuarios_respondidos":  resposta['usuarios_respondidos']
    }
    resultados.append(novo_registro)
with open('output/metricas_questoes_com_dificuldade.csv', mode='w', newline='') as arquivo_csv:
    campos = resultados[0].keys()
    escritor_csv = csv.DictWriter(arquivo_csv, fieldnames=campos)
    escritor_csv.writeheader()
    escritor_csv.writerows(resultados)

print("Dados combinados e salvos em output/metricas_questoes_com_dificuldade.csv")

In [ ]:
import pandas as pd

def ordenar_question(arquivo_question):
    # Ler o arquivo CSV
    question_df = pd.read_csv(arquivo_question)

    # Ordenar o DataFrame pela coluna 'question'
    question_df_sorted = question_df.sort_values(by='question', ascending=True)

    # Sobrescrever o arquivo original com o DataFrame ordenado
    question_df_sorted.to_csv(arquivo_question, index=False)

    print(f"Arquivo '{arquivo_question}' foi ordenado e salvo com sucesso.")

# Exemplo de uso
arquivo_question = 'output/metricas_questoes_com_dificuldade.csv'
ordenar_question(arquivo_question)


In [ ]:
import pandas as pd

def comparar_e_gerar_csv(arquivo_question, arquivo_metrics, arquivo_saida):
    # Ler os arquivos CSV
    question_df = pd.read_csv(arquivo_question)
    metrics_df = pd.read_csv(arquivo_metrics)

    # Renomear a coluna 'question' para 'question_id' em question_df para facilitar a junção
    question_df.rename(columns={'question': 'question_id'}, inplace=True)

    # Fazer a junção dos DataFrames com base na coluna 'question_id'
    merged_df = pd.merge(metrics_df, question_df, on='question_id', how='inner')

    # Ordenar o DataFrame resultante com base na ordem de 'question_id' do question_df original
    merged_df = merged_df.sort_values(by='question_id')

    # Selecionar as colunas que você deseja manter
    colunas_a_manter = metrics_df.columns.tolist()  # Manter todas as colunas do metrics_df

    # Salvar o DataFrame resultante em um novo arquivo CSV
    merged_df.to_csv(arquivo_saida, columns=colunas_a_manter, index=False)

# Exemplo de uso com uma string "raw"
comparar_e_gerar_csv(
    'output/metricas_questoes_com_dificuldade.csv', 
    r'codebench-analytics-full/output/data/code_metrics_professor.csv', #Etapa_4/codebench-analytics-full/output
    'output/dataset_questoes_consolidado.csv'
)

print("Arquivo gerado: output/dataset_questoes_consolidado.csv")



# Ler o arquivo CSV
df = pd.read_csv('output/dataset_questoes_consolidado.csv')

# Renomear a coluna 'question_id' para 'question'
df.rename(columns={'question_id': 'question'}, inplace=True)

# Salvar o DataFrame com o novo nome da coluna
df.to_csv('output/dataset_questoes_consolidado.csv', index=False)

print("Nome da coluna alterado com sucesso no arquivo 'output/dataset_questoes_consolidado.csv'.")
